In [2]:
# Import pipeline
from transcription_pipeline import nuclear_pipeline
from transcription_pipeline import preprocessing_pipeline

from transcription_pipeline import spot_pipeline
from transcription_pipeline import fullEmbryo_pipeline

from transcription_pipeline.spot_analysis import compile_data
from transcription_pipeline.utils import plottable

import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import gc


In [5]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '002'

embryo_list = {
    '001': [
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
    ],
    '002': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],
    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ]
}


In [6]:
def import_dataset(test_dataset_name, use_filtered=False):
    import_previous_ms2 = os.path.isdir(test_dataset_name + '/collated_dataset')
    if import_previous_ms2:
        print('Reading previous imported dataset')
    else:
        print('No previous dataset import found; importing from scratch')

    dataset = preprocessing_pipeline.DataImport(
        name_folder=test_dataset_name,
        trim_series=True,
        working_storage_mode='zarr',
        import_previous=import_previous_ms2,
        use_filtered=use_filtered,
    )
    if not import_previous_ms2:
        dataset.save()
    return dataset

def import_fullEmbryo_dataset(test_dataset_name):
    import_previous_fullEmbryo = os.path.isdir(test_dataset_name + '/preprocessed_full_embryo')

    if import_previous_fullEmbryo:
        print('Reading previous imported FullEmbryo dataset')
    else:
        print('No previous FullEmbryo dataset import found; importing from scratch')

    FullEmbryo_dataset = preprocessing_pipeline.FullEmbryoImport(
        name_folder=test_dataset_name,
        import_previous=import_previous_fullEmbryo,
    )
    if not import_previous_fullEmbryo:
        FullEmbryo_dataset.save()
    return FullEmbryo_dataset

def get_nuclear_params(dataset, nuclear_channel):
    return dict(
        data=dataset.channels_full_dataset[nuclear_channel],
        global_metadata=dataset.export_global_metadata[nuclear_channel],
        frame_metadata=dataset.export_frame_metadata[nuclear_channel],
        series_splits=dataset.series_splits,
        series_shifts=dataset.series_shifts,
        search_range_um=1.5,
        stitch=False,
        stitch_max_distance=4,
        stitch_max_frame_distance=2,
        client=client,
        keep_futures=False,
    )

def get_spot_params(dataset, spot_channel, Labels, search_range_um, threshold_factor, retrack_after_filter):
    return dict(
        data=dataset.channels_full_dataset[spot_channel],
        global_metadata=dataset.export_global_metadata[spot_channel],
        frame_metadata=dataset.export_frame_metadata[spot_channel],
        labels=Labels,
        expand_distance=3,
        search_range_um=search_range_um,
        retrack_search_range_um=4.5,
        spot_sigma_z_bounds=(0.16, 100),
        spot_sigma_x_y_bounds=(0.052, 100),
        spot_sigmas=[0.86, 0.26, 0.26],
        threshold_factor=threshold_factor,
        memory=3,
        retrack_after_filter=retrack_after_filter,
        stitch=True,
        min_track_length=0,
        series_splits=dataset.series_splits,
        series_shifts=dataset.series_shifts,
        keep_bandpass=False,
        keep_futures=False,
        keep_spot_labels=False,
        evaluate=True,
        retrack_by_intensity=False,
        client=client,
    )


def track_import_nuclei(test_dataset_name, dataset, nuclear_channel, spot_channel, retrack=False,
                        folder_name='nuclear_analysis_results'):
    import_previous_nuclear = os.path.isdir(test_dataset_name + '/' + folder_name)

    if import_previous_nuclear and not retrack:
        print(f'Reading previous nuclear tracking results (retrack={retrack})')
        nuclear_tracking = nuclear_pipeline.Nuclear()
        nuclear_tracking.read_results(name_folder=test_dataset_name, source_folder=folder_name)

    else:
        if import_previous_nuclear and retrack:
            print(f'Previous nuclear tracking detected. Retracking nuclei (retrack={retrack})')
        else:
            print(f'No previous nuclear tracking results found; importing from scratch (retrack={retrack})')

        nuclear_tracking = nuclear_pipeline.Nuclear(**get_nuclear_params(dataset, nuclear_channel))
        nuclear_tracking.track_nuclei(
            working_memory_mode="zarr",
            working_memory_folder=test_dataset_name + '/' + folder_name,
            trackpy_log_path=test_dataset_name + "trackpy_log",
        )
        nuclear_tracking.save_results(name_folder=test_dataset_name, save_array_as=None, output_folder=folder_name)

    return nuclear_tracking

def track_import_spots(test_dataset_name, dataset, nuclear_channel, spot_channel,
                       retrack=False, use_nuclear_tracking=True,
                       search_range_um=2, threshold_factor=1.5, retrack_after_filter=False,
                       folder_name='spot_analysis_results'):
    import_previous_spot = os.path.isdir(test_dataset_name + '/' + folder_name)

    if use_nuclear_tracking:
        nuclear_tracking = track_import_nuclei(
            test_dataset_name, dataset, nuclear_channel=nuclear_channel, spot_channel=spot_channel, retrack=False
        )
        Labels = nuclear_tracking.reordered_labels
    else:
        Labels = None

    if import_previous_spot and not retrack:
        print(f'Load from spot tracking results (retrack={retrack})')
        spot_tracking = spot_pipeline.Spot()
        spot_tracking.read_results(name_folder=test_dataset_name, source_folder=folder_name)

    else:
        if import_previous_spot and retrack:
            print(f'Previous spot tracking detected. Retracking spots (retrack={retrack})')
        else:
            print(f'No previous spot tracking results found; importing from scratch (retrack={retrack})')

        spot_tracking = spot_pipeline.Spot(
            **get_spot_params(dataset, spot_channel, Labels, search_range_um, threshold_factor, retrack_after_filter)
        )
        spot_tracking.extract_spot_traces(
            working_memory_folder=test_dataset_name + '/' + folder_name,
            stitch=True,
            retrack_after_filter=retrack_after_filter,
            trackpy_log_path=test_dataset_name + '/trackpy_log',
            verbose=True,
        )
        spot_tracking.save_results(name_folder=test_dataset_name, save_array_as=None, output_folder=folder_name)

    return spot_tracking


In [7]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(
    host="localhost",
    #scheduler_port=37763,
    threads_per_worker=1,
    n_workers=14,
    memory_limit="6GB",
)

client = Client(cluster)

print(client)

<Client: 'tcp://127.0.0.1:43953' processes=14 threads=14, memory=78.23 GiB>


2025-10-14 11:48:25,349 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,351 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,355 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,358 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,368 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,386 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,395 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,402 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,409 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,414 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,417 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,419 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,422 - distributed.nanny - WARNING - Restarting worker
2025-10-14 11:48:25,430 - distributed.

In [8]:
print(client.dashboard_link)


http://127.0.0.1:8787/status


In [11]:
client.restart()


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 14
Total threads: 14,Total memory: 78.23 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43953,Workers: 14
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 78.23 GiB
Comm: tcp://127.0.0.1:34191,Total threads: 1
Dashboard: http://127.0.0.1:39859/status,Memory: 5.59 GiB
Nanny: tcp://127.0.0.1:33375,


In [12]:
nuclear_channel = 1
spot_channel = 0


In [13]:
gc.collect()
test_dataset_name = dataset_folder + embryo_list[key][3]
print(f'\n{"=" * 60}')
print(f'Dataset Path: {test_dataset_name}')
print(f'{"=" * 60}\n')

try:
    gc.disable()
    # Load the dataset with time-filtered data
    dataset = import_dataset(test_dataset_name, use_filtered=True)

    # Load the full embryo dataset
    FullEmbryo_dataset = import_fullEmbryo_dataset(test_dataset_name)
    gc.enable()
    gc.collect()

    # Track and import nuclei with time-filtered data
    nuclear_tracking = track_import_nuclei(
        test_dataset_name, dataset,
        nuclear_channel=nuclear_channel,
        spot_channel=spot_channel,
        retrack=False,
        folder_name='nuclear_analysis_results_time_filtered',
    )

    # Clear memory
    del nuclear_tracking
    gc.collect()

    # Track and import spots with time-filtered data
    spot_tracking = track_import_spots(
        test_dataset_name, dataset,
        nuclear_channel=nuclear_channel,
        spot_channel=spot_channel,
        retrack=False,
        use_nuclear_tracking=False,
        search_range_um=2,
        threshold_factor=1.8,
        retrack_after_filter=True,
        folder_name='spot_analysis_results_time_filtered',
    )

    # Clear memory before next iteration
    del spot_tracking
    del dataset
    del FullEmbryo_dataset
    gc.collect()

    print(f'\nSuccessfully processed: {test_dataset_name}\n')

except Exception as e:
    print(f'\nERROR processing {test_dataset_name}:')
    print(f'{type(e).__name__}: {str(e)}\n')
    # Try to recover by restarting client
    gc.enable()
    gc.collect()
    try:
        client.restart()
    except:
        pass



Dataset Path: /mnt/Data1/Nick/transcription_pipeline/test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02

Reading previous imported dataset
Reading previous imported FullEmbryo dataset
Reading previous nuclear tracking results (retrack=False)

ERROR processing /mnt/Data1/Nick/transcription_pipeline/test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02:
KeyError: '.zgroup'



In [ ]:
print("Processing complete!")